# Chaining codes: a start-to-end (S2E) simulation

This notebook shows how `SIMBA` hands a beam from one tracking code to the next to build a
**start-to-end** simulation of a whole machine, and how to swap the code used for any section.

The core idea is simple: an accelerator is split into consecutive **sections** (the `files:` blocks
in a `.def` settings file), and each section names its own `code:`. `SIMBA` tracks the first section,
dumps the output distribution at its `end_element`, and feeds that distribution into the next section
whose `start_element` matches. This is what lets you use the *right code for each job*:

| Section | Physics that dominates | Good choice of code |
|---|---|---|
| Photoinjector (cathode → few MeV) | space charge at low energy | **ASTRA** / **GPT** |
| Linac / transport | RF acceleration, optics | **Elegant** / **Ocelot** |
| Bunch compressor chicane | coherent synchrotron radiation (CSR) | **CSRTrack** / **Elegant** |
| FEL undulator line | radiation / gain | **Genesis** |

We build on the openly available
[JFEL](https://github.com/astec-stfc/laura-lattices/tree/main/JFEL) lattice from
[laura-lattices](https://github.com/astec-stfc/laura-lattices), the same lattice used in the
[Loading a lattice](https://simba-accelerator.readthedocs.io/en/latest/loading-a-lattice.html) docs.
As shipped, JFEL defines an **ASTRA** injector followed by an **Elegant** linac; here we extend that
chain and show how to re-route individual sections to other codes.

### Setup

As in the other examples, `laura-lattices` must be cloned and a `SimCodes` directory prepared
(see the [SimCodes docs](https://simba-accelerator.readthedocs.io/en/latest/SimCodes.html)),
with their locations exported:

```bash
git clone https://github.com/astec-stfc/laura-lattices.git
export LATTICE_LOCATION=$(pwd)/laura-lattices/JFEL
export SIMCODES=/path/to/simcodes/directory
```

Running **ASTRA**, **Elegant**, **CSRTrack** and **Genesis** all require the `SimCodes` directory
(or a `container_runtime`); only the pure-python codes (Ocelot, Cheetah) can run without it.

In [ ]:
import os
import matplotlib.pyplot as plt
import simba.Framework as fw
from simba.Framework import load_directory

framework = fw.Framework(
    simcodes=os.environ["SIMCODES"],
    directory="./s2e_chain",
    master_lattice=os.environ["LATTICE_LOCATION"],
    generator_defaults="jfel.yaml",
    clean=True,
    verbose=False,
)

# scaling controls the macroparticle count at the cathode; keep it a power of 8 for the
# ASTRA/GPT space-charge mesh. Bump to 3-4 for a production-quality run.
scaling = 2

### 1. The chain that already exists in the `.def`

`jfel_combined.def` defines two sequential sections. Loading the settings builds them as tracking
lines; note how `Linac`'s `start_element` continues from `injector400`'s `end_element` — that match
is what makes the beam flow from one code to the next.

```yaml
files:
  injector400:          # ASTRA, from the cathode
    code: astra
    output:
      zstart: 0
      end_element: JFEL-S02-SIM-APER-01
  Linac:                # Elegant, continues from the injector
    code: elegant
    output:
      start_element: JFEL-S02-SIM-APER-01   # <- matches the line above
      end_element: JFEL-FEL-SIM-MARK-01
```

In [ ]:
framework.loadSettings("Lattices/jfel_combined.def")
print("Tracking lines in order:", framework.lines)

Set up the beam generator (the ASTRA cathode distribution) and enable collective effects on the
linac: longitudinal space charge (LSC) everywhere, and CSR through the bunch-compressor chicane.

In [ ]:
framework.change_generator("ASTRA")
framework.generator.load_defaults("jfel_400_3ps")
framework.generator.thermal_emittance = 0.0005
framework.generator.number_of_particles = 2 ** (3 * scaling)

framework["Linac"].lsc_enable = True
framework["Linac"].csr_enable = True

Track the whole existing chain: ASTRA injector → Elegant linac. `startfile`/`endfile` select the
range of `framework.lines` to run (here, everything from the generator to the end of the linac).

In [ ]:
framework.track(startfile="generator", endfile="Linac")

fwdir = load_directory("./s2e_chain", beams=True)
fwdir.plot(xkey="z", ykeys=["sigma_x", "sigma_y"], ykeys2=["sigma_z"])

### 2. Swapping the code for a section with `change_Lattice_Code`

Because the lattice is defined once (in LAURA) and translated per-code, any section can be re-routed
to a different code at runtime — no change to the lattice itself. This is the quickest way to
cross-check a result, or to move a section onto a code better suited to its physics.

For example, track the linac in **Ocelot** instead of Elegant. `change_Lattice_Code` takes the line
name and the target code (see
[`change_Lattice_Code`](https://simba-accelerator.readthedocs.io/en/latest/Framework.html)).

In [ ]:
# Re-route the Linac onto Ocelot and re-track just that section, continuing from the injector output.
framework.change_Lattice_Code("Linac", "ocelot")
framework.setSubDirectory("./s2e_chain_ocelot_linac")
framework.track(startfile="generator", endfile="Linac")

fwdir_oc = load_directory("./s2e_chain_ocelot_linac", beams=True)
fwdir_oc.plot(xkey="z", ykeys=["sigma_x", "sigma_y"], ykeys2=["sigma_z"])

> **Note.** `change_Lattice_Code("All", code)` swaps every section at once — handy for a quick
> single-code sanity run of a whole machine. See the
> [code cross-comparison](https://simba-accelerator.readthedocs.io/en/latest/) idea for using this
> to benchmark codes against each other.

### 3. Extending the chain: adding CSRTrack and Genesis sections

To make the chain longer you add more `files:` blocks. Each new block continues from the previous
one exactly like `Linac` continues from `injector400`. Two natural additions for an FEL machine:

1. a dedicated **CSRTrack** section for the magnetic chicane, where CSR is strongest, and
2. a **Genesis** section for the undulator line, to model FEL gain.

You can either add these blocks to the `.def` file on disk, or build them in-python with
`FrameworkSettings`. The `.def` route is shown below as the pattern to copy.

```yaml
files:
  injector400:
    code: astra
    output: {zstart: 0, end_element: JFEL-S02-SIM-APER-01}
  Linac_pre:                       # Elegant up to the chicane
    code: elegant
    output: {start_element: JFEL-S02-SIM-APER-01, end_element: JFEL-VBC-MAG-DIP-01}
  chicane:                         # CSRTrack through the bunch compressor
    code: csrtrack
    output: {start_element: JFEL-VBC-MAG-DIP-01, end_element: JFEL-VBC-MAG-DIP-04}
  Linac_post:                      # Elegant from the chicane to the undulator
    code: elegant
    output: {start_element: JFEL-VBC-MAG-DIP-04, end_element: JFEL-FEL-SIM-MARK-01}
  undulator:                       # Genesis FEL line
    code: genesis
    output: {start_element: JFEL-FEL-SIM-MARK-01, end_element: JFEL-FEL-SIM-MARK-02}
```

> **TODO before running:** the element names above (`JFEL-VBC-MAG-DIP-*`, the `JFEL-FEL-*` undulator
> markers) must exist in your lattice's `summary.yaml`. The shipped JFEL lattice contains the chicane
> dipoles; confirm it also contains an undulator line before adding the Genesis block, or point these
> at the equivalent elements in your own lattice.

The in-python equivalent, if you would rather not edit the `.def`, is to assemble a `files` dict and
pass it via `FrameworkSettings` (see the *Option A* setup in the
[Getting started](https://simba-accelerator.readthedocs.io/en/latest/getting-started.html) docs).
Once the sections exist, tracking the full chain is a single call:

In [ ]:
# framework.loadSettings("Lattices/jfel_s2e.def")   # a .def containing the 5 blocks above
# framework.track(startfile="generator", endfile="undulator")
# fwdir = load_directory("./s2e_chain", beams=True)
# fwdir.plot(xkey="z", ykeys=["sigma_x", "sigma_y"], ykeys2=["sigma_z"])

### Cleanup

In [ ]:
import shutil
for d in ("./s2e_chain", "./s2e_chain_ocelot_linac"):
    shutil.rmtree(d, ignore_errors=True)

### Recap

* An S2E simulation is a chain of `files:` sections, each with its own `code:`; the beam flows
  wherever one section's `end_element` matches the next section's `start_element`.
* `change_Lattice_Code(line, code)` re-routes a section to another code at runtime, with no change
  to the lattice definition.
* Lengthening the chain (e.g. adding a **CSRTrack** chicane or a **Genesis** undulator line) is just
  adding more blocks in the same pattern.